In [ ]:
# h2o_ai/health_advisor.py

import requests
import concurrent.futures
from typing import Dict, Optional
import sys
import os

# ============================================
# LOCATION SERVICE (Multi-API Consensus)
# ============================================
class LocationService:
    
    def __init__(self):
        self.cached_location = None
    
    def get_location(self) -> Dict:
        if self.cached_location:
            return self.cached_location
        location = self._get_consensus_location()
        if location:
            self.cached_location = location
            return location
        return self._get_fallback_location()
    
    def _get_consensus_location(self) -> Optional[Dict]:
        apis = [self._try_ipapi, self._try_ipinfo, self._try_ipapi_co]
        try:
            import geocoder
            apis.append(self._try_geocoder)
        except ImportError:
            pass
        
        results = []
        with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
            futures = [executor.submit(api) for api in apis]
            for future in concurrent.futures.as_completed(futures):
                try:
                    result = future.result()
                    if result and result.get('city') != 'Unknown':
                        results.append(result)
                except:
                    pass
        
        if not results:
            return None
        
        city_votes = {}
        for r in results:
            city = r['city']
            city_votes[city] = city_votes.get(city, 0) + 1
        
        best_city = max(city_votes, key=city_votes.get)
        confidence = f"{city_votes[best_city]}/{len(results)} APIs agreed"
        
        for r in results:
            if r['city'] == best_city:
                r['confidence'] = confidence
                return r
        return results[0]
    
    def _try_ipapi(self) -> Optional[Dict]:
        try:
            response = requests.get('http://ip-api.com/json/', timeout=5, headers={'User-Agent': 'H2O-AI/1.0'})
            data = response.json()
            if data.get('status') == 'success':
                return {'lat': data.get('lat'), 'lon': data.get('lon'), 'city': data.get('city', 'Unknown'), 'state': data.get('regionName', ''), 'country': data.get('country', 'Unknown'), 'ip': data.get('query', ''), 'source': 'ip-api'}
        except:
            pass
        return None
    
    def _try_ipinfo(self) -> Optional[Dict]:
        try:
            response = requests.get('https://ipinfo.io/json', timeout=5, headers={'User-Agent': 'H2O-AI/1.0'})
            data = response.json()
            if 'loc' in data:
                lat, lon = data['loc'].split(',')
                return {'lat': float(lat), 'lon': float(lon), 'city': data.get('city', 'Unknown'), 'state': data.get('region', ''), 'country': data.get('country', 'Unknown'), 'ip': data.get('ip', ''), 'source': 'ipinfo'}
        except:
            pass
        return None
    
    def _try_geocoder(self) -> Optional[Dict]:
        try:
            import geocoder
            g = geocoder.ip('me')
            if g.ok and g.city:
                return {'lat': g.latlng[0] if g.latlng else None, 'lon': g.latlng[1] if g.latlng else None, 'city': g.city, 'state': g.state or '', 'country': g.country or 'Unknown', 'ip': g.ip, 'source': 'geocoder'}
        except:
            pass
        return None
    
    def _try_ipapi_co(self) -> Optional[Dict]:
        try:
            response = requests.get('https://ipapi.co/json/', timeout=5, headers={'User-Agent': 'H2O-AI/1.0'})
            data = response.json()
            if 'error' not in data and data.get('city'):
                return {'lat': data.get('latitude'), 'lon': data.get('longitude'), 'city': data.get('city', 'Unknown'), 'state': data.get('region', ''), 'country': data.get('country_name', 'Unknown'), 'ip': data.get('ip', ''), 'source': 'ipapi.co'}
        except:
            pass
        return None
    
    def _get_fallback_location(self) -> Dict:
        return {'lat': 19.0760, 'lon': 72.8777, 'city': 'Mumbai', 'state': 'Maharashtra', 'country': 'India', 'source': 'fallback'}


# ============================================
# WEATHER SERVICE
# ============================================
def get_weather_by_coords(lat, lon):
    params = {
        "latitude": lat,
        "longitude": lon,
        "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,pressure_msl,uv_index",
        "timezone": "auto"
    }
    try:
        response = requests.get("https://api.open-meteo.com/v1/forecast", params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        current = data.get("current", {})
        return {
            "temperature": current.get("temperature_2m"),
            "humidity": current.get("relative_humidity_2m"),
            "wind_speed": current.get("wind_speed_10m"),
            "pressure": current.get("pressure_msl"),
            "uv_index": current.get("uv_index")
        }
    except Exception as e:
        print(f"❌ Weather error: {e}")
        return None


# ============================================
# OUTPUT TO BOTH CONSOLE AND FILE
# ============================================
class Output:
    """Writes output to both console and file"""
    def __init__(self, filename="h2o_ai_report.txt"):
        self.filename = filename
        self.file = open(filename, 'w', encoding='utf-8')
    
    def print(self, text=""):
        """Print to both console and file"""
        # Force flush stdout
        sys.stdout.write(text + "\n")
        sys.stdout.flush()
        # Write to file
        self.file.write(text + "\n")
        self.file.flush()
    
    def close(self):
        self.file.close()
        print(f"\n📄 Full report saved to: {os.path.abspath(self.filename)}")


# ============================================
# MAIN APPLICATION
# ============================================
def main():
    out = Output()
    
    try:
        out.print("=" * 60)
        out.print("   💧  H2O AI - Intelligent Health & Hydration Advisor  💧")
        out.print("=" * 60)
        
        # Step 1: Location
        out.print("\n📍 STEP 1: DETECTING YOUR LOCATION")
        out.print("-" * 40)
        
        location_service = LocationService()
        location = location_service.get_location()
        
        out.print(f"   🏙️  City:      {location['city']}")
        out.print(f"   🏛️  State:     {location.get('state', 'N/A')}")
        out.print(f"   🌍 Country:   {location['country']}")
        out.print(f"   📍 Coordinates: {location['lat']}, {location['lon']}")
        out.print(f"   🌐 Source:    {location.get('source', 'unknown')}")
        if location.get('confidence'):
            out.print(f"   🎯 Confidence: {location['confidence']}")
        
        # Step 2: Weather
        out.print("\n🌤️  STEP 2: FETCHING WEATHER DATA")
        out.print("-" * 40)
        
        weather = get_weather_by_coords(location["lat"], location["lon"])
        
        if not weather:
            out.print("   ❌ Could not retrieve weather data.")
            out.close()
            return
        
        out.print(f"   🌡️  Temperature: {weather['temperature']}°C")
        out.print(f"   💧 Humidity: {weather['humidity']}%")
        out.print(f"   💨 Wind: {weather['wind_speed']} km/h")
        out.print(f"   🌬️  Pressure: {weather['pressure']} hPa")
        out.print(f"   ☀️  UV Index: {weather['uv_index']}")
        
        # Step 3: Recommendations
        out.print("\n📊 STEP 3: GENERATING RECOMMENDATIONS")
        out.print("-" * 40)
        
        temp = weather['temperature']
        humidity = weather['humidity']
        uv = weather['uv_index']
        
        # Weather classification
        if temp >= 30:
            heat = "HOT"
        elif temp >= 20:
            heat = "WARM"
        elif temp >= 10:
            heat = "COOL"
        else:
            heat = "COLD"
        
        if humidity >= 70:
            humid = "HUMID"
        elif humidity >= 40:
            humid = "NORMAL"
        else:
            humid = "DRY"
        
        out.print(f"   ✅ Weather classified as: {heat} & {humid}")
        
        # Water calculation
        base_water = 2.5
        if temp >= 30:
            base_water += 0.6
        elif temp >= 20:
            base_water += 0.3
        elif temp <= 10:
            base_water -= 0.2
        
        if humidity >= 70:
            base_water += 0.3
        elif humidity < 40:
            base_water -= 0.1
        
        water = round(base_water, 1)
        
        # Print full report
        out.print("\n" + "=" * 60)
        out.print("📋 COMPLETE HEALTH RECOMMENDATIONS")
        out.print("=" * 60)
        
        out.print(f"\n1. WEATHER CONDITIONS")
        out.print(f"   Classification: {heat} & {humid}")
        out.print(f"   Temperature: {temp}°C")
        out.print(f"   Humidity: {humidity}%")
        out.print(f"   UV Index: {uv}")
        
        out.print(f"\n2. HYDRATION RECOMMENDATION")
        out.print(f"   Daily water intake: {water} litres ({water*1000:.0f} ml)")
        if water > 3.0:
            out.print(f"   ⚠️  HIGH NEED - Drink water every 15-20 minutes")
            out.print(f"   💡 Carry a water bottle with you always")
        elif temp < 15 and humidity < 40:
            out.print(f"   💡 MODERATE - Cold & dry air can dehydrate silently")
            out.print(f"   💡 Don't skip water even if not feeling thirsty")
        else:
            out.print(f"   ✅ NORMAL - Drink water every 30-45 minutes")
        
        out.print(f"\n3. FOOD RECOMMENDATIONS")
        if heat == "HOT" and humid == "HUMID":
            out.print(f"   🍚 Light meals: khichdi, steamed vegetables, soups")
            out.print(f"   🥒 Hydrating foods: cucumber, watermelon, oranges")
            out.print(f"   🥣 Probiotics: curd, buttermilk, fermented pickles")
            out.print(f"   🚫 Avoid: heavy, oily, spicy dishes")
        elif heat == "HOT":
            out.print(f"   🍉 Water-rich fruits: watermelon, muskmelon, berries")
            out.print(f"   🥗 Light salads, coconut water, fruit juices")
            out.print(f"   🧉 Electrolytes: lassi, buttermilk, lemon water")
            out.print(f"   🚫 Reduce: caffeine and alcohol")
        elif heat == "WARM":
            out.print(f"   🥗 Fresh seasonal vegetables")
            out.print(f"   🥛 Light dairy and proteins")
            out.print(f"   🥒 Hydrating snacks: cucumber, tomatoes")
            out.print(f"   🚫 Avoid: excessive fried or processed food")
        elif heat == "COOL":
            out.print(f"   🍲 Warm meals: soups, stews, porridge")
            out.print(f"   🥕 Root vegetables: carrots, sweet potatoes")
            out.print(f"   🍎 Seasonal fruits: apples, pears, pomegranates")
            out.print(f"   🥛 Warm milk with turmeric/ginger")
        else:
            out.print(f"   🍲 Hot nourishing meals: soups, khichdi, broths")
            out.print(f"   🥜 Warming nuts: sesame, almonds, walnuts")
            out.print(f"   🍚 Whole grains: millets (ragi, bajra), oats")
            out.print(f"   🧉 Herbal teas: ginger-tulsi, kadha, haldi-doodh")
            out.print(f"   🚫 Avoid: cold, raw, or icy foods")
        
        out.print(f"\n4. SKINCARE ADVICE")
        tips_given = False
        if uv >= 6:
            out.print(f"   🧴 Apply SPF 50+ sunscreen every 2 hours")
            tips_given = True
        elif uv >= 3:
            out.print(f"   🧴 Use SPF 30+ sunscreen when going out")
            tips_given = True
        if humidity < 40:
            out.print(f"   💧 Use a good moisturizer (air is dry)")
            tips_given = True
        elif humidity > 80:
            out.print(f"   🌫️ Use light, non-comedogenic products (humid)")
            tips_given = True
        if temp > 35:
            out.print(f"   ☀️ Apply cooling gel or aloe vera")
            tips_given = True
        elif temp < 10:
            out.print(f"   ❄️ Use thicker cream to protect skin")
            tips_given = True
        if not tips_given:
            out.print(f"   😊 Regular skincare routine is sufficient")
        
        out.print(f"\n5. ACTIVITY GUIDANCE")
        if temp > 35:
            out.print(f"   🏊‍♂️ Best: Swimming, indoor yoga, gym")
            out.print(f"   ⚠️  Avoid outdoor exercise between 11 AM - 4 PM")
        elif temp > 30:
            out.print(f"   🚴‍♂️ Good: Morning/evening walks, light jogging")
            out.print(f"   💡 Stay in shade and carry water")
        elif temp > 20:
            out.print(f"   🏃‍♂️ Perfect weather for outdoor activities!")
            out.print(f"   ✅ Running, sports, hiking - all good")
        elif temp > 10:
            out.print(f"   🧘‍♀️ Brisk walking, outdoor yoga, light sports")
        else:
            out.print(f"   🏋️‍♂️ Indoor exercises preferred")
            out.print(f"   💡 Warm-up properly if going outside")
        
        # Hydration schedule
        total_ml = water * 1000
        if temp > 35:
            interval = 15
            serving = 150
        elif temp > 30:
            interval = 20
            serving = 180
        elif temp > 20:
            interval = 30
            serving = 200
        else:
            interval = 45
            serving = 200
        
        servings = min(int(total_ml / serving), 12)
        
        out.print(f"\n6. HOURLY HYDRATION SCHEDULE")
        out.print(f"   Goal: {total_ml:.0f}ml in {servings} servings")
        out.print(f"   Amount: {serving}ml every {interval} minutes")
        out.print("")
        for i in range(servings):
            hour = 8 + (i * interval) // 60
            minute = (i * interval) % 60
            ampm = "AM" if hour < 12 else "PM"
            hour_12 = hour if hour <= 12 else hour - 12
            out.print(f"   🥤 {hour_12:02d}:{minute:02d} {ampm} → Drink {serving}ml water")
        
        out.print("\n" + "=" * 60)
        out.print("   💙 Stay healthy, stay hydrated! - H2O AI")
        out.print("=" * 60)
        
    except Exception as e:
        out.print(f"\n❌ ERROR: {e}")
        import traceback
        out.print(traceback.format_exc())
    
    finally:
        out.close()


if __name__ == "__main__":
    main()


   💧  H2O AI - Intelligent Health & Hydration Advisor  💧

📍 STEP 1: DETECTING YOUR LOCATION
----------------------------------------
   City:      Coimbatore
   State:     Tamil Nadu
   Country:   India
   Coords:    11.0102, 76.9701

🌤️  STEP 2: FETCHING WEATHER DATA
----------------------------------------
   Temperature: 23.6°C
   Humidity:    99%
   Wind:        11.1 km/h
   Pressure:    1010.1 hPa
   UV Index:    0.05

📊 STEP 3: GENERATING RECOMMENDATIONS
----------------------------------------
   Weather: WARM & HUMID
   Water: 3.1 litres/day

📋 HEALTH RECOMMENDATIONS

1. WEATHER CONDITIONS
   WARM & HUMID
   Temp: 23.6°C | Humidity: 99% | UV: 0.05

2. HYDRATION
   Drink 3.1 litres (3100ml) today
   HIGH NEED - Drink every 15-20 minutes

3. FOOD SUGGESTIONS
   Fresh seasonal vegetables
   Light dairy and proteins
   Hydrating snacks: cucumber, tomatoes
   Avoid: excessive fried food

4. SKINCARE
   Use light products (humid)

5. ACTIVITY
   Perfect for outdoor activities!
   Ru